# Chapter 9 — Video Segmentation with the Watershed Algorithm
**Project:** PTVNDM  
**Notebook:** `notebooks/04_watershed_segmentation.ipynb`

---

## Conceptual Background

The **Watershed algorithm** treats a grayscale image as a *topographic surface* — bright pixels are mountain peaks, dark pixels are valleys. Imagine slowly flooding this landscape from every local minimum simultaneously. Where two separate floods would meet, we build a dam — those dams are the **segmentation boundaries**.

The challenge with a naive flood-fill is *over-segmentation* (too many tiny regions from noise). The solution is **marker-based Watershed**: we pre-label regions we are *certain* about (sure foreground and sure background) and let the algorithm only decide the ambiguous boundaries in between.

### Pipeline at a Glance
```
Raw Frame
  └─► Grayscale  ──► Otsu Threshold  ──► Morphological Cleaning
                                              │
                                    ┌─────────┴──────────┐
                               Sure BG (dilate)   Sure FG (distanceTransform)
                                    └─────────┬──────────┘
                                          Unknown Region
                                              │
                                    connectedComponents → Markers
                                              │
                                        cv2.watershed
                                              │
                                    Boundary = –1  ──► Coloured Red
```

## 1 · Imports & Configuration

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")

# ── Paths ──────────────────────────────────────────────────────────────────────
# All paths are relative to notebooks/ so they work on every machine
# that clones the PTVNDM repository without path adjustments.
VIDEO_PATH  = "../data/raw/dance/dance3.mp4"
OUTPUT_DIR  = "../outputs"
OUTPUT_IMG  = os.path.join(OUTPUT_DIR, "segmentation_res.jpg")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Watershed hyper-parameters ─────────────────────────────────────────────────
MORPH_KERNEL_SIZE   = (3, 3)   # kernel for morphological operations
OPEN_ITERATIONS     = 2        # MORPH_OPEN  – removes small white noise dots
DILATE_ITERATIONS   = 3        # dilate      – expands sure-background region
DIST_FG_THRESHOLD   = 0.7      # keep only pixels with dist > 70 % of max
                                # (higher = more conservative foreground mask)

print(f"\nInput  : {os.path.abspath(VIDEO_PATH)}")
print(f"Output : {os.path.abspath(OUTPUT_IMG)}")

## 2 · Load Video & Extract First Usable Frame

In [ ]:
# Open the video file with OpenCV's VideoCapture.
# VideoCapture works for both files and live streams (use integer index for webcam).
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(
        f"Cannot open video: '{VIDEO_PATH}'.\n"
        "Verify the file exists at PTVNDM/data/raw/dance/dance3.mp4"
    )

# ── Video metadata ─────────────────────────────────────────────────────────────
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video opened successfully.")
print(f"  Resolution : {width} × {height} px")
print(f"  FPS        : {fps}")
print(f"  Frames     : {total_frames}")
print(f"  Duration   : {total_frames / fps:.1f} s")

## 3 · Watershed Pipeline (Step-by-Step)

The helper function below encapsulates the full algorithm. Each stage is annotated with its *topographic mapping* interpretation so the logic is traceable end-to-end.

In [ ]:
def apply_watershed(frame: np.ndarray) -> tuple[np.ndarray, dict]:
    """
    Apply the marker-based Watershed algorithm to a single BGR frame.

    Parameters
    ----------
    frame : np.ndarray
        Input BGR image (as returned by cap.read()).

    Returns
    -------
    result : np.ndarray
        A copy of `frame` with watershed boundaries drawn in red.
    intermediates : dict
        Intermediate images for visualisation / debugging.
    """
    intermediates = {}

    # ── Step 1 · Grayscale conversion ──────────────────────────────────────────
    # Reduce the 3-channel colour image to a single intensity channel.
    # In topographic terms: this grayscale image IS our elevation map –
    # bright regions are peaks, dark regions are valleys.
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    intermediates["1_gray"] = gray

    # ── Step 2 · Otsu's thresholding ───────────────────────────────────────────
    # THRESH_BINARY_INV : foreground objects become WHITE (255), background BLACK.
    # THRESH_OTSU       : OpenCV automatically finds the optimal threshold value
    #                     by minimising intra-class intensity variance (Otsu 1979).
    # The combined flag means: apply Otsu's method, then invert the result.
    _, thresh = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )
    intermediates["2_otsu_thresh"] = thresh

    # ── Step 3 · Noise removal with MORPH_OPEN ─────────────────────────────────
    # Morphological OPEN = erosion followed by dilation.
    # Effect: removes isolated white pixels (salt noise) that are smaller
    # than the kernel, without shrinking the main object contours significantly.
    # In topographic terms: this smooths small noisy peaks from our flood plains.
    kernel = np.ones(MORPH_KERNEL_SIZE, dtype=np.uint8)
    opening = cv2.morphologyEx(
        thresh, cv2.MORPH_OPEN,
        kernel, iterations=OPEN_ITERATIONS
    )
    intermediates["3_opening"] = opening

    # ── Step 4 · Sure background via dilation ──────────────────────────────────
    # Dilation expands the white foreground region outward.
    # After enough iterations the expanded mask is GUARANTEED to include all
    # foreground pixels plus a safe margin → this is our "sure background" label.
    # Topographic analogy: we are raising the water level high enough to flood
    # everything we are confident is background.
    sure_bg = cv2.dilate(
        opening, kernel,
        iterations=DILATE_ITERATIONS
    )
    intermediates["4_sure_bg"] = sure_bg

    # ── Step 5 · Sure foreground via Distance Transform ────────────────────────
    # distanceTransform assigns to every white pixel its Euclidean distance
    # to the nearest black (background) pixel.
    # Pixels FAR from any background edge are deep inside an object → they
    # are the most certain foreground pixels (the "valley floors" in our map).
    # We keep only pixels whose distance exceeds DIST_FG_THRESHOLD × max_dist,
    # giving us a conservative but reliable sure-foreground mask.
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    _, sure_fg = cv2.threshold(
        dist_transform,
        DIST_FG_THRESHOLD * dist_transform.max(),
        255, cv2.THRESH_BINARY
    )
    sure_fg = np.uint8(sure_fg)      # convert float32 mask → uint8
    intermediates["5_dist_transform"] = dist_transform
    intermediates["5_sure_fg"]        = sure_fg

    # ── Step 6 · Unknown region ────────────────────────────────────────────────
    # Pixels that are in sure_bg but NOT in sure_fg are ambiguous –
    # they could belong to an object OR to the background.
    # These are the regions the Watershed algorithm will resolve for us.
    # Topographic analogy: the shoreline where floodwaters from different
    # basins are about to meet.
    unknown = cv2.subtract(sure_bg, sure_fg)
    intermediates["6_unknown"] = unknown

    # ── Step 7 · Marker creation with connectedComponents ─────────────────────
    # connectedComponents labels each isolated blob in sure_fg with a unique
    # integer (1, 2, 3 …). Label 0 is background.
    # We then:
    #   +1  →  shift all labels up so background becomes 1, objects become ≥2.
    #          This is required because Watershed treats 0 as "unlabelled".
    #   = 0 →  mark the unknown region as 0 so Watershed will fill it in.
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1           # shift: 0→1 (sure BG), 1→2 (first object) …
    markers[unknown == 255] = 0     # tell Watershed: "these pixels are unknown"
    intermediates["7_markers_before"] = markers.copy()

    # ── Step 8 · Watershed execution ───────────────────────────────────────────
    # cv2.watershed MODIFIES markers in-place.
    # Where two different regions would flood into each other it writes –1,
    # which represents the boundary / dam between segments.
    # Input MUST be a 3-channel (BGR) uint8 image – hence we pass the original frame.
    markers = cv2.watershed(frame, markers)
    intermediates["8_markers_after"] = markers.copy()

    # ── Step 9 · Visualise boundaries ─────────────────────────────────────────
    # Wherever markers == –1 the algorithm found a boundary.
    # We colour those pixels RED on a copy of the original frame.
    result = frame.copy()
    result[markers == -1] = [0, 0, 255]   # BGR → Red

    return result, intermediates

## 4 · Process Frames & Save the First Segmented Result

In [ ]:
original_frame   = None   # will hold the raw frame for side-by-side display
segmented_frame  = None   # will hold the watershed result
debug_data       = None   # intermediate images from the pipeline
frame_index      = 0

print("Processing frames …")

while True:
    ret, frame = cap.read()

    if not ret:
        # End of video or read error – stop processing
        print(f"  End of video reached at frame {frame_index}.")
        break

    frame_index += 1

    # Run the full Watershed pipeline on this frame
    result, intermediates = apply_watershed(frame)

    # Save the FIRST successfully processed frame and stop
    # (for a university demo we only need one representative result).
    if segmented_frame is None:
        original_frame  = frame.copy()
        segmented_frame = result.copy()
        debug_data      = intermediates

        # Persist the segmented image to disk
        cv2.imwrite(OUTPUT_IMG, segmented_frame)
        print(f"  ✔ Frame {frame_index} segmented and saved → {OUTPUT_IMG}")
        break   # remove this line to process the entire video

# Always release the capture handle when done
cap.release()
print("VideoCapture released.")

if segmented_frame is None:
    raise RuntimeError("No frame could be segmented. Check the input video.")

## 5 · Side-by-Side Visualisation

In [ ]:
# OpenCV stores images as BGR; matplotlib expects RGB – convert before display.
original_rgb  = cv2.cvtColor(original_frame,  cv2.COLOR_BGR2RGB)
segmented_rgb = cv2.cvtColor(segmented_frame, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Watershed Segmentation — dance3.mp4",
    fontsize=15, fontweight="bold", y=1.01
)

# ── Left panel : Original frame ────────────────────────────────────────────────
axes[0].imshow(original_rgb)
axes[0].set_title("Original Frame", fontsize=13, pad=10)
axes[0].axis("off")

# ── Right panel : Segmented frame ─────────────────────────────────────────────
axes[1].imshow(segmented_rgb)
axes[1].set_title("Segmented Frame (Watershed)", fontsize=13, pad=10)
axes[1].axis("off")

# Add a legend explaining the red boundary colour
boundary_patch = mpatches.Patch(color="red", label="Watershed boundary (marker = –1)")
axes[1].legend(
    handles=[boundary_patch],
    loc="lower right", fontsize=9,
    framealpha=0.8
)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "segmentation_comparison.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Comparison plot saved to ../outputs/segmentation_comparison.png")

## 6 · Pipeline Intermediate Stages (Debug View)

This cell visualises each intermediate output so you can verify every step of the topographic mapping pipeline.

In [ ]:
stages = [
    ("1_gray",          "Step 1 · Grayscale\n(elevation map)"),
    ("2_otsu_thresh",   "Step 2 · Otsu Threshold\n(binary foreground)"),
    ("3_opening",       "Step 3 · Morph OPEN\n(noise removed)"),
    ("4_sure_bg",       "Step 4 · Sure Background\n(dilated)"),
    ("5_sure_fg",       "Step 5 · Sure Foreground\n(distance threshold)"),
    ("6_unknown",       "Step 6 · Unknown Region\n(BG – FG)"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle(
    "Watershed Pipeline — Intermediate Stages",
    fontsize=14, fontweight="bold"
)

for ax, (key, title) in zip(axes.flat, stages):
    img = debug_data[key]

    # Distance transform is float32 – normalise to [0, 255] for display
    if img.dtype == np.float32:
        display = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        cmap = "hot"        # heat map conveys distance gradient intuitively
    else:
        display = img
        cmap = "gray"

    ax.imshow(display, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "segmentation_pipeline_steps.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Pipeline debug plot saved to ../outputs/segmentation_pipeline_steps.png")

## 7 · Marker Map Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Marker Map: Before vs After Watershed", fontsize=13, fontweight="bold")

# markers_before: shows the pre-labelled regions sent into cv2.watershed
# markers_after : shows the resolved regions (–1 = boundary)
for ax, key, title in [
    (axes[0], "7_markers_before", "Step 7 · Markers BEFORE\n(connectedComponents + shift)"),
    (axes[1], "8_markers_after",  "Step 8 · Markers AFTER\n(cv2.watershed result; –1 = boundary)"),
]:
    mdata = debug_data[key].astype(np.float32)
    # Normalise for display; -1 boundary pixels become 0 (black) after this.
    disp  = cv2.normalize(mdata, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    ax.imshow(disp, cmap="nipy_spectral")
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, "segmentation_markers.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Marker map saved to ../outputs/segmentation_markers.png")

## 8 · Output Summary

In [ ]:
print("=" * 55)
print("OUTPUT SUMMARY")
print("=" * 55)

files = [
    "segmentation_res.jpg",
    "segmentation_comparison.png",
    "segmentation_pipeline_steps.png",
    "segmentation_markers.png",
]

for fname in files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  ✔ {fname:<45} {size_kb:>7.1f} KB")
    else:
        print(f"  ✘ {fname} — NOT FOUND")

print("=" * 55)

# Count how many distinct regions the algorithm identified
markers_after  = debug_data["8_markers_after"]
num_regions    = markers_after.max()   # label 1 = background; ≥2 = objects
num_boundaries = np.sum(markers_after == -1)

print(f"\nWatershed statistics (frame {frame_index}):")
print(f"  Distinct regions identified : {num_regions}")
print(f"  Boundary pixels (marker –1) : {num_boundaries:,}")